# Every cycle, handed on

**Temporal-stage notebook 6 in `notebooks/temporal/`.** `notebooks/temporal/README.md` says what the notebooks in this folder share. This notebook uses the three numbers of `03_window_summary_fd001.ipynb` — the newest reading, the average and the change — so read that one first.

## What this notebook is for

The stage in this notebook, `raw_window`, hands each sensor's node all 30 of its readings, untouched. It learns nothing and changes nothing. What changes is the layer after it: the first graph layer's weight matrix becomes 30 rows by 32 instead of Table 4's 1 by 32, and its rows multiply cycles. So that layer becomes 32 learned ways of weighing a sensor's 30 cycles, the same 32 for every sensor — a temporal stage with learned numbers, built inside a layer that is not named one.

This was the network this project trained until 2026-08-28, when it was found and removed (assumption A-07). It is kept, behind a gate, so that the comparison which decided that change can be run again at today's settings, and so that a run using it says so in its name.

## What it does not do

- **It does not train anything**, and it reads no trained network: no stored five-seed run of this stage exists under this folder's settings. Step 5 says so and prints the command that would make one.
- **It says nothing about how well this stage predicts.** Without a stored run there is no result, and the results of the removed mode, from before 2026-08-28, were measured under other settings.
- **It does not compare this stage with another.** `07_comparison_fd001.ipynb` does that, for the stages that have stored runs.

## The notebook in one list

| # | Step | What goes in | What comes out |
|---|------|--------------|----------------|
| 1 | **Load the window** | `config/fd001.yaml`, seed 42 | the held-back engine's last window, 14 sensors by 30 cycles |
| 2 | **Run the stage** | the window | the same 30 numbers per sensor |
| 3 | **What the first graph layer does with them** | T50's neighbours' 30 readings, three ways of weighing cycles, and a trained window-summary network | each way worked by hand, and the trained network's first layer rebuilt as a raw-window one |
| 4 | **The gate and the cost** | the network built with this stage | why it is built only when named, and its learned numbers |
| 5 | **Read the stored results** | `reports/` | none under this folder's settings, and the command that makes one |

## 1. Background

### What the first graph layer computes

The first graph layer computes `Â X W(1)`, then applies ReLU, which replaces every negative number with zero: the paper's Equation 8 (page 6). Everything this notebook follows happens before the ReLU, in `Â X W(1)`:

- `X` is what the temporal stage hands on: one row per sensor, 14 rows, and as many columns as the stage hands each sensor.
- `Â` is the rescaled sensor graph, 14 by 14. Row `i` holds a weight for every sensor joined to sensor `i`, itself included, and the weights of a row add up to 1 (assumption A-21). `Â X` replaces each sensor's row by the weighted average of its neighbours' rows: the **mixing**.
- `W(1)` has one row per column of `X` and one column per output, 32 outputs here. Multiplying by it turns each mixed row into 32 numbers: the **weighing**. Its numbers are learned, and the same `W(1)` serves all 14 sensors.

The code does it in that order, mixing first (`src/thesis_gnn_rul_xai/models/layers.py`).

With one number per sensor, `X` is 14 by 1, `W(1)` is 1 by 32, and output `c` of sensor `i` is the mixed number times `W(1)[0, c]`: 32 rescalings of one number. With 30 numbers per sensor, `X` is 14 by 30, `W(1)` is 30 by 32, and output `c` of sensor `i`, before the ReLU, is

```
output[i, c]  =  Σ over t of  m_i(t) × W(1)[t, c]          m_i(t) = Σ over j of Â[i, j] × x_j(t)
```

- `t` is the position in the window, 1 to 30, and `x_j(t)` is sensor `j`'s reading there;
- `m_i(t)` is sensor `i`'s mixed reading at position `t`: the weighted average, over `i` and its neighbours `j`, of their readings at that one position;
- `W(1)[t, c]` is the learned multiplier of position `t` in output `c`.

So each output is a weighted sum over the 30 positions of the mixed readings: column `c` of `W(1)` is a way of weighing cycles, learned, and shared by all 14 sensors. That is a temporal stage — and the reason this one is built only when named.

**A tiny example**, three positions and one output. Mixed readings `m = [1, 2, 4]` and the column `[0, 0, 1]` give `0 × 1 + 0 × 2 + 1 × 4 = 4`, the newest reading. The column `[1/3, 1/3, 1/3]` gives `(1 + 2 + 4) / 3 = 7/3`, the average. The column `[−1, 0, 1]` gives `−1 + 0 + 4 = 3`, the rise from the first position to the last. One layer, three columns, three of the summaries other stages hand on.

**Do not confuse** this stage, which learns nothing, with the network built on it, which learns how to weigh the cycles in its first graph layer. The stage's own count of learned numbers is zero; the cost is in the layer after it.

## 2. Setup

In [ ]:
from __future__ import annotations

import json
import logging
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import yaml
from IPython.display import display

# The project's top folder is the nearest one above this notebook that holds
# pyproject.toml. Adding its src/ folder to Python's search path makes the
# package importable from a clone that was never installed.
PROJECT_ROOT = next(
    folder for folder in (Path.cwd(), *Path.cwd().parents)
    if (folder / "pyproject.toml").is_file()
)
sys.path.insert(0, str(PROJECT_ROOT / "src"))

try:
    import torch
except ImportError as missing:
    raise ImportError(
        "This notebook builds the network's temporal stage, which needs PyTorch, and PyTorch is "
        'not installed. Install the project with  pip install -e ".[model]"'
    ) from missing

from thesis_gnn_rul_xai.config import load_config
from thesis_gnn_rul_xai.data.pipeline import run_preprocessing
from thesis_gnn_rul_xai.models import ModelConfig, build_model
from thesis_gnn_rul_xai.models import ESTGNN
from thesis_gnn_rul_xai.models.temporal import RawWindow, WindowSummary
from thesis_gnn_rul_xai.training import evaluate_model, load_seed, read_run
from thesis_gnn_rul_xai.training import TrainingConfig, find_runs
from thesis_gnn_rul_xai.utils.logging_setup import configure_logging

configure_logging(logging.ERROR)
pd.set_option("display.width", 120)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_colwidth", None)    # a run's folder name is printed whole
plt.rcParams.update({"figure.dpi": 110, "font.size": 9})

print(f"Project root: {PROJECT_ROOT}")
print(f"PyTorch     : {torch.__version__}")

## 3. The window

### Step 1 — Load the window

**What.** Rebuild the data from `config/fd001.yaml`, take the held-back engine with the most windows, and take its last window. Print the 30 readings of T50, the temperature at the outlet of the low-pressure turbine, the sensor every notebook in this folder follows.

**Why.** Every notebook here works on this same window, so a difference between two notebooks is a difference between two stages. `notebooks/temporal/README.md` says why this engine, this window and this sensor, and what the units of a reading are.

**Check.** The window must be 14 sensors by 30 cycles, come from an engine the training never saw, and end at the cycle the engine failed.

In [ ]:
config = load_config(PROJECT_ROOT / "config" / "fd001.yaml")
prepared = run_preprocessing(config)

sensors = list(prepared.sensor_symbols)

# The held-back engine with the most windows, and its last window.
held_back = prepared.validation
units, counts = np.unique(held_back.units, return_counts=True)
ENGINE = int(units[np.argmax(counts)])
engine_rows = np.flatnonzero(held_back.units == ENGINE)
engine_rows = engine_rows[np.argsort(held_back.end_cycles[engine_rows])]

SENSOR = "T50"
s = sensors.index(SENSOR)
window = held_back.features[engine_rows[-1]]             # (14 sensors, 30 cycles)
end_cycle = int(held_back.end_cycles[engine_rows[-1]])
cycles = np.arange(end_cycle - window.shape[1] + 1, end_cycle + 1)
readings = window[s]                                     # T50's 30 readings, oldest first

print(f"Engine {ENGINE}: {len(engine_rows)} windows, held back from training under seed {config.seed}.")
print(f"Worked window: cycles {cycles[0]} to {cycles[-1]}, shape {window.shape}, "
      f"remaining life at its end {held_back.targets[engine_rows[-1]]:.0f} cycles.\n")
display(pd.DataFrame([readings], index=[f"{SENSOR} reading"],
                     columns=pd.MultiIndex.from_arrays([range(1, 31), cycles],
                                                       names=["position", "engine cycle"])).round(6))

window_checks = {
    "the window is 14 sensors by 30 cycles": window.shape == (14, 30),
    "its engine is not a training engine": ENGINE not in set(prepared.train.units),
    "it ends at the cycle the engine failed": float(held_back.targets[engine_rows[-1]]) == 0.0,
}
assert all(window_checks.values()), window_checks
print("OK  " + "; ".join(window_checks))

## 4. The stage, step by step

### Step 2 — Run the stage

**What.** Hand the window to `RawWindow(30)` as a batch of one.

**Why.** To see that the stage itself does nothing: everything this notebook is about happens in the layer after it.

**Check.** The output must be the window itself, 14 sensors by 30 cycles, number for number.

In [ ]:
stage = RawWindow(window_size=30)
with torch.no_grad():
    handed_on = stage(torch.from_numpy(window[None]))      # (1, 14, 30)
print(f"in  {tuple(window[None].shape)}    out {tuple(handed_on.shape)}")
stage_changes_nothing = bool(np.array_equal(handed_on[0].numpy(), window))
assert stage_changes_nothing
print(f"OK  each sensor's node receives its 30 readings unchanged; {SENSOR}'s last is "
      f"{float(handed_on[0, s, -1]):.6f}, as in the window.")

### Step 3 — What the first graph layer does with 30 numbers

**What.** Three parts, all for T50.

1. Mix: from the rescaled graph's row for T50, take each of T50's neighbours' 30 readings and average them, position by position, into T50's 30 mixed readings `m(t)`.
2. Weigh, by hand, with three columns of `W(1)` written down rather than learned: all weight on position 30; 1/30 on every position; and the change of `03_window_summary_fd001.ipynb`, which weighs position `t` by `29 × (t − 15.5) / 2247.5`. Put the same three columns into the first graph layer of a network built with this stage and check that the layer gives the same three numbers.
3. Load the trained network of a stored window-summary run, seed 42, and rebuild its first graph layer as a raw-window one: column `c` of the new `W(1)` is the trained layer's three multipliers spread over the 30 positions by those three columns.

**Why.** The first two parts show, on real readings, what a column of a 30-row `W(1)` is: a way of weighing cycles, of which the newest reading, the average and the change are three. The third shows it on trained numbers: every one of the trained window-summary layer's 32 outputs is one such column, so a raw-window layer can compute exactly what that trained layer computes. The opposite does not hold — 30 rows of 32 hold 960 multipliers, three rows 96 — so the raw-window layer can also weigh cycles in ways no three-number summary can.

**Check.** The loaded network must score the test engines as the run recorded for seed 42, to within 0.0001 cycles; the hand mixing must match the graph; each hand weighing must match the layer's output; the three columns must give the same numbers as the graph mixing what the last-cycle, window-average and window-summary stages hand on; and the rebuilt layer must give the trained layer's 32 outputs for all 14 sensors.

In [ ]:
A = prepared.adjacency_normalized                         # (14, 14), each row adds up to 1
row = A[s]
neighbours = [sensors[j] for j in np.flatnonzero(row)]
print(f"Row {SENSOR} of the rescaled graph: " + ", ".join(f"{sensors[j]} {row[j]:.4f}" for j in np.flatnonzero(row))
      + f"   (adds up to {row.sum():.4f})\n")

mixed = row @ window.astype(np.float64)                  # (30,) T50's mixed reading at every position
table = pd.DataFrame(window[np.flatnonzero(row)].astype(np.float64), index=neighbours,
                     columns=pd.MultiIndex.from_arrays([range(1, 31), cycles], names=["position", "engine cycle"]))
table.loc[f"mixed, {SENSOR}"] = mixed
display(table.iloc[:, [0, 1, 2, 27, 28, 29]].round(6))
t = 0
print("position 1, by hand: " + " + ".join(f"{row[j]:.1f} × {window[j, t]:+.6f}" for j in np.flatnonzero(row))
      + f" = {mixed[t]:+.6f}\n")
mixing_matches = bool(np.allclose(mixed, (A @ window.astype(np.float64))[s]))

positions = np.arange(1, 31)
columns = {
    "newest reading": (positions == 30).astype(np.float64),
    "average": np.full(30, 1 / 30),
    "change": 29 * (positions - 15.5) / 2247.5,
}
by_hand = {name: float(mixed @ weights) for name, weights in columns.items()}
for name, weights in columns.items():
    shown = " + ".join(f"({weights[k]:+.4f} × {mixed[k]:+.4f})" for k in (0, 1))
    print(f"{name:15s} {shown} + ... + ({weights[29]:+.4f} × {mixed[29]:+.4f}) = {by_hand[name]:+.6f}")

network = build_model(A, ModelConfig(temporal="raw_window"), window_size=config.window_size, rul_max=config.rul_max)
first_layer = network.graph_layers[0]
with torch.no_grad():
    first_layer.weight.zero_()
    for c, weights in enumerate(columns.values()):
        first_layer.weight[:, c] = torch.from_numpy(weights.astype(np.float32))
    layer_out = first_layer(torch.from_numpy(window[None]), network.adjacency)[0, s, :3].double().numpy()
layer_matches = bool(np.allclose(layer_out, list(by_hand.values()), atol=1e-5))

with torch.no_grad():
    summary = WindowSummary(30)(torch.from_numpy(window[None]))[0].double().numpy()   # (14, 3)
other_stages_match = bool(np.allclose((A @ summary)[s], list(by_hand.values()), atol=1e-5))
print(f"\nthe raw-window layer, with those three columns: {np.round(layer_out, 6).tolist()}")
print(f"the graph mixing the window summary's three numbers: {np.round((A @ summary)[s], 6).tolist()}")
assert mixing_matches and layer_matches and other_stages_match
print("OK  the hand mixing matches the graph; each hand weighing matches the layer; the three columns give "
      "what the last-cycle, window-average and window-summary stages hand on, mixed.")

**Reading the output.** In the paper's graph T50 is joined to T30, Nc, W31 and W32 (its Table 3); with T50 itself that makes five sensors, and dividing the row by its total gives each 0.2. So T50's mixed reading at a position is the plain average of five sensors' readings there: at position 1, 0.2 × (0.447597 + 1.292245 + 2.797086 − 0.829339 − 1.228228) = 0.495872.

The three columns then weigh the 30 mixed readings. The first keeps position 30's 0.994222, the second averages all thirty to 0.759712, and the third, the change, gives 0.519178. The layer with those three columns gives the same three numbers, and so does the graph mixing the newest reading, the average and the change of each sensor: what the last-cycle, window-average and window-summary stages hand on. Those stages and this one differ in where the weighing of the cycles is decided. There, it is fixed before training, by the stage; here, training learns it, in `W(1)`.

In [ ]:
SUMMARY_RUN = PROJECT_ROOT / "reports" / "fd001_physics_row_cap_5seed_varsplit_window_summary_cfgba82e3_5602773-dirty_20260914T233435Z"
if not (SUMMARY_RUN / "summary.json").is_file():
    raise FileNotFoundError(
        f"{SUMMARY_RUN.name} is not in reports/. It is the stored five-seed run of the window-summary "
        "stage. A run with the same settings is made with\n\n"
        "    python scripts/train.py --config config/fd001.yaml --temporal window_summary --seeds 5\n\n"
        "after which SUMMARY_RUN above must name the folder it writes."
    )
summary_run = read_run(SUMMARY_RUN)
summary_network, summary_data = load_seed(summary_run, 42)
recorded_rmse = float(summary_run.results.set_index("seed").loc[42, "rmse"])
scored_rmse = evaluate_model(summary_network, summary_data.test)["rmse"]
same_network = abs(scored_rmse - recorded_rmse) < 1e-4
assert same_network
print(f"Stored run : {SUMMARY_RUN.relative_to(PROJECT_ROOT).as_posix()}, commit {summary_run.summary['commit']}")
print(f"OK  seed 42's network scores the test engines at {scored_rmse:.6f} cycles here; the run recorded "
      f"{recorded_rmse:.6f} ({scored_rmse - recorded_rmse:+.1e}).\n")
trained = summary_network.graph_layers[0].weight.detach().double().numpy()        # (3, 32)
spread = np.stack(list(columns.values()), axis=1)                                  # (30, 3)
rebuilt = spread @ trained                                                          # (30, 32)
print(f"Trained window-summary W(1): {trained.shape}; the three columns of position weights: {spread.shape}; "
      f"rebuilt raw-window W(1): {rebuilt.shape}")
c = 0
print(f"\noutput 1 of the trained layer weighs newest, average and change by "
      f"{trained[0, c]:+.4f}, {trained[1, c]:+.4f}, {trained[2, c]:+.4f}.")
print("Spread over the positions, its weight on position t is "
      f"{trained[0, c]:+.4f} × [t = 30] + {trained[1, c]:+.4f} / 30 + {trained[2, c]:+.4f} × 29 × (t − 15.5) / 2247.5:")
display(pd.DataFrame({"weight on the position": rebuilt[:, c]}, index=pd.Index(positions, name="position")).T.round(5))

with torch.no_grad():
    first_layer.weight.copy_(torch.from_numpy(rebuilt.astype(np.float32)))
    as_raw = first_layer(torch.from_numpy(window[None]), network.adjacency)[0].double().numpy()      # (14, 32)
    carried = summary_network.temporal(torch.from_numpy(window[None]))
    as_trained = summary_network.graph_layers[0](carried, summary_network.adjacency)[0].double().numpy()
same_graph = bool(np.allclose(summary_data.adjacency_normalized, A))
rebuilt_matches = bool(np.allclose(as_raw, as_trained, atol=1e-5))
assert same_graph and rebuilt_matches
print(f"\nOK  on the worked window the rebuilt raw-window layer gives the trained layer's 32 outputs for all "
      f"14 sensors (largest gap {np.abs(as_raw - as_trained).max():.1e}); {SENSOR}'s output 1 is {as_trained[s, c]:+.6f}.")

figure, axis = plt.subplots(figsize=(8, 3.2))
for k in range(4):
    axis.plot(positions, rebuilt[:, k], "o-", markersize=3, label=f"output {k + 1}")
axis.axhline(0, color="0.7", linewidth=0.8)
axis.set(title="Four of the trained window-summary layer's 32 outputs, as weights on positions",
         xlabel="position in the window", ylabel="weight on the position")
axis.legend()
figure.tight_layout()
plt.show()

**Reading the figure.** Output 1 of the trained window-summary layer weighs the newest reading by +0.3029, the average by +0.4076 and the change by +0.1761. Spread over the positions, that is a weight of −0.01937 on position 1, rising by 0.1761 × 29 / 2247.5 = 0.00227 a position to 0.04427 on position 29, then 0.34942 on position 30, where the newest reading's +0.3029 is added to the line. Every one of the trained layer's 32 outputs has that shape — a straight line across positions 1 to 29, with position 30 lifted or lowered off it — because the three columns it is built from are a constant, a straight line and a single last position. A raw-window layer could learn any of these 32 columns, and any other shape as well: it has 960 multipliers to learn where the window-summary layer has 96.

### Step 4 — The gate, and what it costs

**What.** Try to build the network the way a caller might by accident, handing `ESTGNN` a `RawWindow` directly; then build it through `build_model` with the stage named, as `scripts/train.py --temporal raw_window` does, and count its learned numbers.

**Why.** The network asks whether its stage holds learned numbers. A stage that hands on more than one number and learns nothing is putting cycles into the first graph layer, so it is refused unless named, and a run that names it says so in its folder name and its `summary.json`.

**Check.** The direct build must be refused; the named build must give `W(1)` of shape `(30, 32)`, no learned numbers in the stage, and 61,729 + 29 × 32 = 62,657 in the whole network.

In [ ]:
TABLE_4_NAMES = {"graph_layers.0": "GCN-1", "graph_layers.1": "GCN-2",
                 "graph_layers.2": "GCN-3", "fc1": "FC-1", "fc2": "FC-2"}


def parameter_table(network):
    """Learned numbers in each part of a network, the temporal stage first."""
    parts = {"temporal stage": 0}
    for name, parameter in network.named_parameters():
        part = "temporal stage" if name.startswith("temporal.") else TABLE_4_NAMES[name.rsplit(".", 1)[0]]
        parts[part] = parts.get(part, 0) + parameter.numel()
    parts["whole network"] = sum(parts.values())
    return pd.DataFrame({"learned numbers": parts})


try:
    ESTGNN(A, temporal=RawWindow(30), rul_max=config.rul_max)
    refused = False
except ValueError as refusal:
    refused = True
    print("Refused:", str(refusal).split(". ")[0] + ".\n")

network = build_model(A, ModelConfig(temporal="raw_window"), window_size=config.window_size, rul_max=config.rul_max)
costs = parameter_table(network)
display(costs.T)
cost_checks = {
    "the direct build is refused": refused,
    "the stage learns nothing": int(costs.loc["temporal stage", "learned numbers"]) == 0,
    "W(1) is (30, 32)": tuple(network.graph_layers[0].weight.shape) == (30, 32),
    "the whole network holds 62,657": int(costs.loc["whole network", "learned numbers"]) == 61_729 + 29 * 32,
}
assert all(cost_checks.values()), cost_checks
print("OK  " + "; ".join(cost_checks))

The 928 learned numbers this stage adds over Table 4's network, 29 rows of 32, are all in the first graph layer, and every one of them weighs a position in the window. Runs of the removed arrangement had 62,657. The default network's count, 61,729, is now checked by a test, because that one number would have shown the change at once (finding F-06).

### Step 5 — The stored results of this stage

**What.** Look for stored five-seed runs of this stage under this folder's settings, and list the runs of the removed arrangement that are still in `reports/`, read from their own files.

**Why.** Without a stored run this stage has no result here, and the command that would make one is the useful output. The old runs are listed so that their numbers are not mistaken for one.

In [ ]:
stored = [
    run
    for run in find_runs(preprocessing=config, seeds=5, split="varies_with_seed", early_stopping=True)
    if run.model_config == ModelConfig(temporal="raw_window")
]
print(f"Stored five-seed runs of raw_window under config/fd001.yaml's settings: {len(stored)}")
if not stored:
    print("Make one with\n\n    python scripts/train.py --config config/fd001.yaml --temporal raw_window --seeds 5\n")

old = []
for folder in sorted((PROJECT_ROOT / "reports").glob("fd001_*")):
    recorded = yaml.safe_load((folder / "preprocessing_config.yaml").read_text(encoding="utf-8")) \
        if (folder / "preprocessing_config.yaml").is_file() else {}
    if recorded.get("node_feature_mode") != "window":
        continue
    summary = json.loads((folder / "summary.json").read_text(encoding="utf-8"))
    old.append({"run, under reports/": folder.name, "commit": summary["commit"],
                "graph": recorded["graph"], "graph rescaling": recorded["adjacency_normalization"],
                "sensor scaling": recorded["normalization"], "learning-rate path": summary.get("lr_schedule"),
                "test RMSE": f"{summary['mean']['rmse']:.3f} ± {summary['std']['rmse']:.3f}"})
print(f"Runs of the arrangement removed on 2026-08-28 still in reports/: {len(old)}")
display(pd.DataFrame(old).set_index("run, under reports/"))
print(f"This folder's settings: graph {config.graph}, graph rescaling {config.adjacency_normalization}, "
      f"sensor scaling {config.normalization}, learning-rate path {TrainingConfig().lr_schedule}.")
old_runs_differ = all(run["sensor scaling"] != config.normalization for run in old)

**No stored result.** Nothing in `reports/` measures this stage under this folder's settings, so this notebook has no number for it, and `07_comparison_fd001.ipynb` has no column for it.

The four runs of the removed arrangement were made on 2026-08-27, at `cdcc827-dirty`. They rescaled each sensor to a fixed range, separately in each operating condition (`minmax_per_condition`), not by subtracting the training average and dividing by the spread, and lowered the learning rate along the exponential path, not by a single drop; three of them also rescaled the graph by the degrees of both ends of each edge ("symmetric"), not by each row's total. Their saved preprocessing names a setting the code no longer reads, so `find_runs` skips them. Each describes the network of its own settings, and none of them is this stage's result today: the design for swappable stages asks for this stage to be measured again at today's settings rather than carried forward (`docs/DESIGN_swappable_stages.md`, its step 6).

## Final verification

In [ ]:
checks = [
    ("The window is the folder's window", all(window_checks.values()), f"engine {ENGINE}, cycle {end_cycle}"),
    ("The stage hands on the window unchanged", stage_changes_nothing, "14 sensors by 30 cycles"),
    ("The hand mixing matches the graph", mixing_matches, f"{SENSOR} and {len(neighbours) - 1} neighbours"),
    ("Each hand weighing matches the layer", layer_matches, "newest, average, change"),
    ("The columns give the other stages' numbers", other_stages_match, "newest, average, change, mixed"),
    ("The loaded network scores what the run recorded", same_network,
     f"{scored_rmse - recorded_rmse:+.1e} cycles"),
    ("The rebuilt layer gives the trained layer's outputs", rebuilt_matches, "14 sensors by 32 outputs"),
    ("The gate and the cost", all(cost_checks.values()), "refused unnamed; 62,657 learned numbers"),
    ("The old runs used other sensor scaling", old_runs_differ, f"{len(old)} run(s)"),
]
width = max(len(name) for name, _, _ in checks) + 2
print(f"{'CHECK':<{width}} RESULT   DETAIL")
print("-" * (width + 40))
for name, passed, detail in checks:
    print(f"{name:<{width}} {'PASS' if passed else 'FAIL':<8} {detail}")
print("-" * (width + 40))
failed = [name for name, passed, _ in checks if not passed]
if failed:
    raise AssertionError(f"{len(failed)} check(s) failed: {failed}")
print(f"All {len(checks)} checks passed.")
print(f"\nStored five-seed runs of this stage under this folder's settings: {len(stored)}; the step on stored results lists them, or prints the command that makes one.")

## Summary

### What this notebook produced

- **The stage's output**: the window, unchanged.
- **What the first graph layer then does**, worked on T50's real neighbours: mix them position by position, then weigh the 30 positions by a learned column. The newest reading, the average and the change are three such columns, and a trained window-summary layer's 32 outputs are 32 more.
- **The gate**: built only when named, because the weighing of cycles happens in a layer not named a temporal stage.
- **Its cost**: no learned numbers in the stage, 928 more than Table 4's network in the first graph layer, 62,657 in all.
- **No stored result** under this folder's settings, and the command that makes one.

### Decisions that were not read from the paper

| Decision | Value used | Written up as |
|---|---|---|
| Keeping this stage at all, behind a gate | built only when named | A-07 |
| Each row of the rescaled graph adding up to 1 | divided by its total | A-21 |
| Which window and which sensor are shown | the held-back engine's last window, T50 | `notebooks/temporal/README.md` |

### References

- Wen, J.-R., et al. (2026). Explainable spatiotemporal graph neutral network with dynamic SHAP enhancement for aeroengine RUL prediction. *Aerospace Science and Technology*, 168, 110875. Equation 8 and Table 4.